
# Store Cleaned Data in SQL

●Create the database

●Create tables based on the ER diagram

●Load cleaned data into SQL

●Apply primary and foreign key constraints

●Verify the data after loading


## 1. Create SQL Tables

In [1]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


# 1: SQL Connection

In [2]:
import mysql.connector

conn_mysql = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Rg1815$$"

)

cursor_mysql = conn_mysql.cursor()
print("MySQL connection established!")

MySQL connection established!


# 2. Create Database in MySQl

In [3]:
cursor_mysql.execute("CREATE DATABASE IF NOT EXISTS Cart2insight_db;")
print("MySQl database Cart2insight_db created successfully!")

MySQl database Cart2insight_db created successfully!


# 3. Creating a table in the Database

Now that we have a database, we need to create a table a to store the 

- customers
- orders
- order_items
- products
- sellers
- payments
- reviews
- category_translation
- geolocation


# Creating a Table in MySQl

In [4]:
cursor_mysql = conn_mysql.cursor()

In [5]:
cursor_mysql.execute("USE Cart2insight_db;")

In [6]:
cursor_mysql.execute("SELECT DATABASE();")
print(cursor_mysql.fetchone()) # which table use this db

('cart2insight_db',)


In [6]:
table_queries = [

"""
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state VARCHAR(10)
)
""",

"""
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    order_status VARCHAR(30),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME
)
""",

"""
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price DECIMAL(10,2),
    freight_value DECIMAL(10,2)
)
""",

"""
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g DECIMAL(10,2),
    product_length_cm DECIMAL(10,2),
    product_height_cm DECIMAL(10,2),
    product_width_cm DECIMAL(10,2)
)
""",

"""
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state VARCHAR(10)
)
""",

"""
CREATE TABLE IF NOT EXISTS payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(50),
    payment_installments INT,
    payment_value DECIMAL(10,2)
)
""",

"""
CREATE TABLE IF NOT EXISTS reviews (
    review_id VARCHAR(50),
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME
)
""",

"""
CREATE TABLE IF NOT EXISTS category_translation (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
)
""",

"""
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(10)
)
"""
]

for query in table_queries:
    cursor_mysql.execute(query)

conn_mysql.commit()

print("All 9 tables created successfully!")

All 9 tables created successfully!


In [7]:
cursor_mysql.execute("SHOW TABLES;")

tables = cursor_mysql.fetchall()

for table in tables:
    print(table[0])

category_translation
customers
geolocation
order_items
orders
payments
products
reviews
sellers


# 4 Inserting Data Into the Table 

# Step 1 Load The Customers Data
- Load the cleaned Customers CSV

In [8]:
import pandas as pd

customers_clean = pd.read_csv(
    "../Data/2_cleaned/customers_clean.csv"
)


In [10]:
customers_clean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
insert_customers = """
INSERT INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)
"""

Step 3 — Insert the data

In [10]:
customer_data = list(
    customers_clean[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ].itertuples(index=False, name=None)
)

cursor_mysql.executemany(
    insert_customers,
    customer_data
)

conn_mysql.commit()

print("Customers data inserted successfully!")

IntegrityError: 1062 (23000): Duplicate entry '06b8999e2fba1a1fbc88172c00ba8bc7' for key 'customers.PRIMARY'

In [11]:
cursor_mysql.execute("TRUNCATE TABLE customers;")
conn_mysql.commit()

ProgrammingError: 1701 (42000): Cannot truncate a table referenced in a foreign key constraint (`Cart2insight_db`.`orders`, CONSTRAINT `fk_orders_customer`)

In [12]:
cursor_mysql.execute("SELECT COUNT(*) FROM customers;")
print(cursor_mysql.fetchone())

(99441,)


# Step 2 Load The orders Data
- Load the cleaned Customers CSV

In [15]:
orders_clean = pd.read_csv(
    "../Data/2_cleaned/orders_clean.csv"
)

orders_clean.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [16]:
orders_clean["order_id"].duplicated().sum()

np.int64(0)

insert query:

In [17]:
insert_orders = """
INSERT INTO orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

In [18]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

insert the orders_data data into MySQL.

In [19]:
orders_data = list(
    orders_clean[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].itertuples(index=False, name=None)
)

In [20]:
# Check the dates stored inside the MySQL orders table

cursor_mysql.execute("""
SELECT
    order_purchase_timestamp,
    order_delivered_customer_date,
    DATEDIFF(
        order_delivered_customer_date,
        order_purchase_timestamp
    ) AS delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
LIMIT 10;
""")

check_data = cursor_mysql.fetchall()

check_df = pd.DataFrame(
    check_data,
    columns=[
        "Purchase Date",
        "Delivered Date",
        "Delivery Days"
    ]
)

check_df

,Purchase Date,Delivered Date,Delivery Days
0,2017-09-13 08:59:02,2017-09-20 23:43:48,7
1,2017-04-26 10:53:06,2017-05-12 16:04:24,16
2,2018-01-14 14:33:31,2018-01-22 13:19:16,8
3,2018-08-08 10:00:35,2018-08-14 13:32:39,6
4,2017-02-04 13:57:51,2017-03-01 16:42:31,25
5,2017-05-15 21:42:34,2017-05-22 13:44:35,7
6,2017-12-10 11:53:48,2017-12-18 22:03:38,8
7,2018-07-04 12:08:27,2018-07-09 14:04:07,5
8,2018-03-19 18:40:33,2018-03-29 18:17:31,10
9,2018-07-02 13:59:39,2018-07-04 17:28:31,2


In [21]:
cursor_mysql.execute("SELECT COUNT(*) FROM orders;")
print(cursor_mysql.fetchone())

(99441,)


# Step 3 Load The order_items Data
- Load the cleaned Customers CSV

In [87]:
order_items_clean = pd.read_csv(
    "../Data/2_cleaned/order_items_clean.csv"
)

order_items_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


composite key for duplicates:

In [97]:
order_items_clean[
    ["order_id", "order_item_id"]
].duplicated().sum()

np.int64(0)

Then convert the date column:

In [89]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

In [231]:
conn_mysql.rollback()

cursor_mysql.execute("TRUNCATE TABLE order_items;")
conn_mysql.commit()

print("Order Items table cleared.")

Order Items table cleared.


In [232]:
cursor_mysql.execute("SELECT COUNT(*) FROM order_items;")
print(cursor_mysql.fetchone())

(0,)


insert the Order Items data into MySQL.

In [233]:
cursor_mysql.executemany(
    insert_order_items,
    order_items_data
)

conn_mysql.commit()

In [234]:
cursor_mysql.execute("SELECT COUNT(*) FROM order_items;")
print(cursor_mysql.fetchone())

(112650,)


insert query:

In [100]:
insert_order_items = """
INSERT INTO order_items (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

insert the Order Items data into MySQL.

load the cleaned Products file:

In [93]:
products_clean = pd.read_csv(
    "../Data/2_cleaned/products_clean.csv"
)

products_clean.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


Because Products still has a few missing numeric values, convert missing values to None before sending them to MySQL:

Now create the insert query:

In [104]:
products_clean = products_clean.astype(object).where(
    pd.notna(products_clean),
    None
)

In [105]:
insert_products = """
INSERT INTO products (
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [145]:
conn_mysql.rollback()

convert missing values to SQL NULL

In [144]:
products_clean = products_clean.astype(object).where(
    pd.notna(products_clean),
    None
)

In [146]:
products_data = list(
    products_clean[
        [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].itertuples(index=False, name=None)
)

In [142]:
cursor_mysql.execute("SELECT COUNT(*) FROM products;")
print(cursor_mysql.fetchone())

(32951,)


# Step 5 Load The sellers Data
- Load the cleaned Customers CSV

In [113]:
sellers_clean = pd.read_csv(
    "../Data/2_cleaned/sellers_clean.csv"
)

sellers_clean.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


Create the insert query:

In [115]:
insert_sellers = """
INSERT INTO sellers (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
VALUES (%s, %s, %s, %s)
"""

Prepare the rows:

In [116]:
sellers_data = list(
    sellers_clean[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ].itertuples(index=False, name=None)
)

Insert them:

In [117]:
cursor_mysql.executemany(
    insert_sellers,
    sellers_data
)

conn_mysql.commit()

print("Sellers data inserted successfully!")

Sellers data inserted successfully!


# Step 6 Load The payments Data
- Load the cleaned Customers CSV

cleaned Payments file:

In [151]:
payments_clean = pd.read_csv(
    "../Data/2_cleaned/payments_clean.csv"
)
payments_clean .head()


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


check the composite key:

In [157]:
payments_clean[
    ["order_id", "payment_sequential"]
].duplicated().sum()

np.int64(0)

Create the insert query:

In [154]:
payments_data = """
INSERT INTO payments (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
VALUES (%s, %s, %s, %s, %s)
"""

In [163]:
conn_mysql.rollback()

Data Insert

In [164]:
payments_data = list(
    payments_clean[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ].itertuples(index=False, name=None)
)

In [166]:
cursor_mysql.executemany(
    insert_payments,
    payments_data
)

conn_mysql.commit()

In [165]:
type(payments_data)

list

cursor_mysql.execute("SELECT COUNT(*) FROM payments;")
print(cursor_mysql.fetchone())

In [162]:
cursor_mysql.execute("SELECT COUNT(*) FROM payments;")
print(cursor_mysql.fetchone())

(0,)


# Step 7 Load The reviews Data
- Load the cleaned Customers CSV

In [167]:
# load the cleaned Reviews file:
reviews_clean = pd.read_csv(
    "../Data/2_cleaned/reviews_clean.csv"
)
reviews_clean.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Title,No Comment,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Title,No Comment,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Title,No Comment,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,No Title,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,No Title,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [168]:
# Now check the composite key:
reviews_clean[
    ["review_id", "order_id"]
].duplicated().sum()

np.int64(0)

review date columns:

In [169]:
reviews_clean["review_creation_date"] = pd.to_datetime(
    reviews_clean["review_creation_date"],
    errors="coerce"
)

reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"],
    errors="coerce"
)

Then create the insert query:

In [170]:
insert_reviews = """
INSERT INTO reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

In [171]:
# prepare the Reviews data
reviews_data = list(
    reviews_clean[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].itertuples(index=False, name=None)
)

In [172]:
# insert
cursor_mysql.executemany(
    insert_reviews,
    reviews_data
)

conn_mysql.commit()

print("Reviews data inserted successfully!")

Reviews data inserted successfully!


# Step 9 Load The geolocation Data
- Load the cleaned Customers CSV

In [174]:

# Load the cleaned file:
category_translation_clean = pd.read_csv(
    "../Data/2_cleaned/category_translation_clean.csv"
)
category_translation_clean.head()


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [180]:
category_translation_clean[
    "product_category_name"
].duplicated().sum()

np.int64(0)

In [211]:
conn_mysql.rollback()

cursor_mysql.execute("TRUNCATE TABLE category_translation;")
conn_mysql.commit()

print("Category table cleared.")

Category table cleared.


compare the CSV rows with the SQL table:

In [212]:
print("CSV rows:", len(category_translation_clean))

cursor_mysql.execute(
    "SELECT COUNT(*) FROM category_translation;"
)
print("SQL rows:", cursor_mysql.fetchone()[0])

CSV rows: 71
SQL rows: 0


In [213]:
cursor_mysql.execute("SELECT COUNT(*) FROM category_translation;")
print(cursor_mysql.fetchone())

(0,)


In [214]:
category_data = list(
    category_translation_clean[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ].itertuples(index=False, name=None)
)



Now insert it one time:

In [215]:
cursor_mysql.executemany(
    insert_category,
    category_data
)

conn_mysql.commit()

print("Category Translation data inserted successfully!")

Category Translation data inserted successfully!


insert query:

In [207]:
insert_category = """
INSERT INTO category_translation (
    product_category_name,
    product_category_name_english
)
VALUES (%s, %s)
"""

In [208]:
# Verify the data 
cursor_mysql.execute(
    "SELECT COUNT(*) FROM category_translation;"
)
print(cursor_mysql.fetchone())

(71,)


# Step 9 Load The geolocation Data
- Load the cleaned Customers CSV

In [217]:
cursor_mysql.execute("SELECT COUNT(*) FROM geolocation;")
print(cursor_mysql.fetchone())

(0,)


In [218]:
insert_geolocation = """
INSERT INTO geolocation (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
VALUES (%s, %s, %s, %s, %s)
"""

In [219]:
for chunk in pd.read_csv(
    "../Data/2_cleaned/geolocation_clean.csv",
    chunksize=5000
):

    geolocation_data = list(
        chunk[
            [
                "geolocation_zip_code_prefix",
                "geolocation_lat",
                "geolocation_lng",
                "geolocation_city",
                "geolocation_state"
            ]
        ].itertuples(index=False, name=None)
    )

    cursor_mysql.executemany(
        insert_geolocation,
        geolocation_data
    )

    conn_mysql.commit()

print("Geolocation data inserted successfully!")

Geolocation data inserted successfully!


In [221]:
cursor_mysql.execute("SELECT COUNT(*) FROM geolocation;")
print(cursor_mysql.fetchone())

(1000163,)


In [222]:
tables = [
    "customers",
    "orders",
    "order_items",
    "products",
    "sellers",
    "payments",
    "reviews",
    "category_translation",
    "geolocation"
]

for table in tables:
    cursor_mysql.execute(f"SELECT COUNT(*) FROM {table};")
    count = cursor_mysql.fetchone()[0]
    print(table, ":", count)

customers : 99441
orders : 99441
order_items : 225300
products : 32951
sellers : 3095
payments : 103886
reviews : 99224
category_translation : 71
geolocation : 1000163


## Step 6.5 Primary Keys and Foreign Keys
This code is adding Primary Keys to three SQL tables.

A Primary Key is a column, or combination of columns, that uniquely identifies each row.

In [235]:
cursor_mysql.execute("""
ALTER TABLE order_items
ADD PRIMARY KEY (order_id, order_item_id);
""")

cursor_mysql.execute("""
ALTER TABLE payments
ADD PRIMARY KEY (order_id, payment_sequential);
""")

cursor_mysql.execute("""
ALTER TABLE reviews
ADD PRIMARY KEY (review_id, order_id);
""")

conn_mysql.commit()

print("Composite primary keys added successfully!")

Composite primary keys added successfully!


# Foreign Keys

First add only the relationship between Customers → Orders



In [ ]:
cursor_mysql.execute("""
ALTER TABLE orders
ADD CONSTRAINT fk_orders_customer
FOREIGN KEY (customer_id)
REFERENCES customers(customer_id);
""")

print("Orders → Customers foreign key added successfully!")

# Every customer_id in orders must already exist in customers.

Orders → Customers foreign key added successfully!


add the foreign key from Order Items → Orders.

In [ ]:
cursor_mysql.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_orderitems_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id);
""")

conn_mysql.commit()

print("Order Items → Orders foreign key added successfully!")

# Every order_id in order_items must already exist in the orders table.

Order Items → Orders foreign key added successfully!


connect Order Items → Products.

In [ ]:
cursor_mysql.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_orderitems_product
FOREIGN KEY (product_id)
REFERENCES products(product_id);
""")

conn_mysql.commit()

print("Order Items → Products foreign key added successfully!")

#Every product used in order_items must exist in the products table.

Order Items → Products foreign key added successfully!


connect Order Items → Sellers.

In [ ]:
cursor_mysql.execute("""
ALTER TABLE order_items
ADD CONSTRAINT fk_orderitems_seller
FOREIGN KEY (seller_id)
REFERENCES sellers(seller_id);
""")

conn_mysql.commit()

print("Order Items → Sellers foreign key added successfully!")

# Every seller in order_items must already exist in the sellers table.

Order Items → Sellers foreign key added successfully!


connect Payments → Orders.

In [ ]:
cursor_mysql.execute("""
ALTER TABLE payments
ADD CONSTRAINT fk_payments_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id);
""")

conn_mysql.commit()

print("Payments → Orders foreign key added successfully!")

# Every payment must belong to an existing order.

Payments → Orders foreign key added successfully!


one is Reviews → Orders.

In [ ]:
cursor_mysql.execute("""
ALTER TABLE reviews
ADD CONSTRAINT fk_reviews_order
FOREIGN KEY (order_id)
REFERENCES orders(order_id);
""")

conn_mysql.commit()

print("Reviews → Orders foreign key added successfully!")

# Every review must belong to an existing order.

Reviews → Orders foreign key added successfully!


Customers → Orders ✅
Orders → Order Items ✅
Products → Order Items ✅
Sellers → Order Items ✅
Orders → Payments ✅
Orders → Reviews ✅

Products → Category Translation ⚠️ No FK because source data has unmatched categories
Geolocation → No FK because ZIP codes are not unique

# Step 6.6: SQL Business Analysis

Business Question 1: How many total orders are there?



In [243]:
cursor_mysql.execute("""
SELECT COUNT(*) AS total_orders
FROM orders;
""")

result = cursor_mysql.fetchone()

print("Total Orders:", result[0])

Total Orders: 99441


# Business Question 2: Orders by Status.
 How many orders are there for each order status?


In [244]:
cursor_mysql.execute("""
SELECT order_status, COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('delivered', 96478)
('shipped', 1107)
('canceled', 625)
('unavailable', 609)
('invoiced', 314)
('processing', 301)
('created', 5)
('approved', 2)


### Business Question 3: What is the total revenue generated from all orders?

Total Revenue.


In [245]:
cursor_mysql.execute("""
SELECT SUM(payment_value) AS total_revenue
FROM payments;
""")

result = cursor_mysql.fetchone()

print("Total Revenue:", result[0])

Total Revenue: 16008872.12


# 4: Revenue by Payment Type

 # How much revenue comes from each payment type?

In [246]:
cursor_mysql.execute("""
SELECT payment_type,
       SUM(payment_value) AS total_revenue
FROM payments
GROUP BY payment_type
ORDER BY total_revenue DESC;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('credit_card', Decimal('12542084.19'))
('boleto', Decimal('2869361.27'))
('voucher', Decimal('379436.87'))
('debit_card', Decimal('217989.79'))
('not_defined', Decimal('0.00'))


# 5: Top 10 Product Categories by Sales.

# Which are the top 10 product categories by sales?

In [247]:
cursor_mysql.execute("""
SELECT
    COALESCE(
        c.product_category_name_english,
        p.product_category_name,
        'unknown'
    ) AS category,
    SUM(oi.price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation c
    ON p.product_category_name = c.product_category_name
GROUP BY category
ORDER BY total_sales DESC
LIMIT 10;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('health_beauty', Decimal('1258681.34'))
('watches_gifts', Decimal('1205005.68'))
('bed_bath_table', Decimal('1036988.68'))
('sports_leisure', Decimal('988048.97'))
('computers_accessories', Decimal('911954.32'))
('furniture_decor', Decimal('729762.49'))
('cool_stuff', Decimal('635290.85'))
('housewares', Decimal('632248.66'))
('auto', Decimal('592720.11'))
('garden_tools', Decimal('485256.46'))


# 6: Top 10 sellers by sales.

# Which are the top 10 sellers by sales?

In [248]:
cursor_mysql.execute("""
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    SUM(oi.price) AS total_sales
FROM order_items oi
JOIN sellers s
    ON oi.seller_id = s.seller_id
GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
ORDER BY total_sales DESC
LIMIT 10;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('4869f7a5dfa277a7dca6462dcf3b52b2', 'guariba', 'SP', Decimal('229472.63'))
('53243585a1d6dc2643021fd1853d8905', 'lauro de freitas', 'BA', Decimal('222776.05'))
('4a3ca9315b744ce9f8e9374361493884', 'ibitinga', 'SP', Decimal('200472.92'))
('fa1c13f2614d7b5c4749cbc52fecda94', 'sumare', 'SP', Decimal('194042.03'))
('7c67e1448b00f6e969d365cea6b010ab', 'itaquaquecetuba', 'SP', Decimal('187923.89'))
('7e93a43ef30c4f03f38b393420bc753a', 'barueri', 'SP', Decimal('176431.87'))
('da8622b14eb17ae2831f4ac5b9dab84a', 'piracicaba', 'SP', Decimal('160236.57'))
('7a67c85e85bb2ce8582c35f2203ad736', 'sao paulo', 'SP', Decimal('141745.53'))
('1025f0e2d44d7041d6cf58b6550e0bfa', 'sao paulo', 'SP', Decimal('138968.55'))
('955fee9216a65b617aa5c0531780ce60', 'sao paulo', 'SP', Decimal('135171.70'))


calculates the total product sales for each seller using:

# sorts from highest to lowest and shows the top 10 sellers.

# 7: Top 10 customers by spending.

it shows the top 10 highest-spending customers.

In [251]:
cursor_mysql.execute("""
SELECT
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    SUM(p.payment_value) AS total_spent
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN payments p
    ON o.order_id = p.order_id
GROUP BY
    c.customer_unique_id,
    c.customer_city,
    c.customer_state
ORDER BY total_spent DESC
LIMIT 10;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('0a0a92112bd4c708ca5fde585afaa872', 'rio de janeiro', 'RJ', Decimal('13664.08'))
('46450c74a0d8c5ca9395da1daac6c120', 'florianopolis', 'SC', Decimal('9553.02'))
('da122df9eeddfedc1dc1f5349a1a690c', 'araruama', 'RJ', Decimal('7571.63'))
('763c8b1c9c68a0229c42c9fc6f662b93', 'vila velha', 'ES', Decimal('7274.88'))
('dc4802a71eae9be1dd28f5d788ceb526', 'campo grande', 'MS', Decimal('6929.31'))
('459bef486812aa25204be022145caa62', 'vitoria', 'ES', Decimal('6922.21'))
('ff4159b92c40ebe40454e3e6a7c35ed6', 'marilia', 'SP', Decimal('6726.66'))
('4007669dec559734d6f53e029e360987', 'divinopolis', 'MG', Decimal('6081.54'))
('5d0a2980b292d049061542014e8960bf', 'goiania', 'GO', Decimal('4809.44'))
('eebb5dda148d3893cdaf5b5ca3040ccb', 'maua', 'SP', Decimal('4764.34'))


# 8: Average Customer Review Score.

#  What is the average customer review score?

In [252]:
cursor_mysql.execute("""
SELECT
    ROUND(AVG(review_score), 2) AS average_review_score
FROM reviews;
""")

result = cursor_mysql.fetchone()

print("Average Review Score:", result[0])

Average Review Score: 4.09


# 9: How many reviews are there for each rating from 1 to 5?

#  How many reviews are there for each rating from 1 to 5?

In [254]:
cursor_mysql.execute("""
SELECT
    review_score,
    COUNT(*) AS total_reviews
FROM reviews
GROUP BY review_score
ORDER BY review_score;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

(1, 11424)
(2, 3151)
(3, 8179)
(4, 19142)
(5, 57328)


# Question 10: Average delivery time in days.

#  What is the average delivery time in days?

In [255]:
cursor_mysql.execute("""
SELECT
    ROUND(
        AVG(
            DATEDIFF(
                order_delivered_customer_date,
                order_purchase_timestamp
            )
        ),
        2
    ) AS average_delivery_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL;
""")

result = cursor_mysql.fetchone()

print("Average Delivery Time:", result[0], "days")

Average Delivery Time: -21952.07 days


# 11: Late Deliveries.

How many orders were delivered later than the estimated delivery date?

In [256]:
cursor_mysql.execute("""
SELECT
    COUNT(*) AS late_deliveries
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_delivered_customer_date > order_estimated_delivery_date;
""")

result = cursor_mysql.fetchone()

print("Late Deliveries:", result[0])

Late Deliveries: 7827


# Question 12: Monthly sales trend.

How does revenue change month by month?

1. DATE_FORMAT(...) changes the order date into year-month format.

2. JOIN payments connects the orders table with the payments table using:

3. SUM(p.payment_value) adds all payment amounts for the same month.

4. GROUP BY month creates one result for each month, and:


In [257]:
cursor_mysql.execute("""
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS month,
    ROUND(SUM(p.payment_value), 2) AS monthly_revenue
FROM orders o
JOIN payments p
    ON o.order_id = p.order_id
GROUP BY month
ORDER BY month;
""")

results = cursor_mysql.fetchall()

for row in results:
    print(row)

('2016-09', Decimal('252.24'))
('2016-10', Decimal('59090.48'))
('2016-12', Decimal('19.62'))
('2017-01', Decimal('138488.04'))
('2017-02', Decimal('291908.01'))
('2017-03', Decimal('449863.60'))
('2017-04', Decimal('417788.03'))
('2017-05', Decimal('592918.82'))
('2017-06', Decimal('511276.38'))
('2017-07', Decimal('592382.92'))
('2017-08', Decimal('674396.32'))
('2017-09', Decimal('727762.45'))
('2017-10', Decimal('779677.88'))
('2017-11', Decimal('1194882.80'))
('2017-12', Decimal('878401.48'))
('2018-01', Decimal('1115004.18'))
('2018-02', Decimal('992463.34'))
('2018-03', Decimal('1159652.12'))
('2018-04', Decimal('1160785.48'))
('2018-05', Decimal('1153982.15'))
('2018-06', Decimal('1023880.50'))
('2018-07', Decimal('1066540.75'))
('2018-08', Decimal('1022425.32'))
('2018-09', Decimal('4439.54'))
('2018-10', Decimal('589.67'))


This query shows how much revenue the business generated in each month.

This is useful because later you can make a line chart and see whether sales are increasing or decreasing over time.